# Working with Image Classification Datasets

`datamaite` loads still-image classification datasets into the neutral
`ImageClassificationDataset`, which satisfies the image-classification
protocol from [MAITE](https://mit-ll-ai-technology.github.io/maite/) --
JATIC's common interface for test-and-evaluation datasets and models -- and
writes them back out through registered writers. This tutorial
builds a tiny synthetic dataset in a temp directory, loads it, and
round-trips it -- so it runs anywhere, with no downloads.

```bash
pip install "datamaite[maite,ic]"
```

The `ic` extra provides OpenCV for image decode; `maite` provides the
protocol check (see [installation](../getting-started/installation.md)).

In [ ]:
import datamaite

# major.minor.patch only, so the docs don't show a hatch-vcs dev suffix in CI
print(f"datamaite {'.'.join(datamaite.__version__.split('.')[:3])}")

## Build a tiny YOLO classification dataset

YOLO/Ultralytics classification is a folder layout -- `<split>/<class>/<image>`,
where the folder name is the label. No config file is needed.

In [ ]:
import atexit
import shutil
import tempfile
from pathlib import Path

import cv2
import numpy as np

from datamaite import load_ic

ic_root = Path(tempfile.mkdtemp(prefix="dm_yolo_ic_"))
atexit.register(shutil.rmtree, ic_root, ignore_errors=True)  # removed when the kernel exits
for cls, shade in (("cat", 30), ("dog", 220)):
    d = ic_root / "train" / cls
    d.mkdir(parents=True, exist_ok=True)
    for i in range(2):
        cv2.imwrite(str(d / f"{cls}_{i}.jpg"), np.full((32, 32, 3), shade, np.uint8))

ds = load_ic(ic_root, dataset_format="yolo")
print("samples     :", ds.sample_count)
print("index2label :", ds.index2label())

## Inspect a sample

Indexing decodes the image lazily (CHW `uint8`) and builds a one-hot target
over the class taxonomy. The dataset satisfies the MAITE
image-classification protocol.

In [ ]:
from maite.protocols import image_classification as ic_protocols

img, target, meta = ds[0]
onehot = np.asarray(target)
print("image shape (CHW):", img.shape, img.dtype)
print("target one-hot   :", onehot.tolist(), "-> class", ds.index2label()[int(onehot.argmax())])
print("metadata         :", meta)
print("MAITE IC Dataset :", isinstance(ds, ic_protocols.Dataset))

## Validate (not yet for IC)

`validate()` covers HMIE only today; for other formats it raises
`NotImplementedError` rather than silently passing:

In [ ]:
from datamaite import DatasetFormat, validate

try:
    validate(ic_root, dataset_format=DatasetFormat.YOLO)
except NotImplementedError as exc:
    print("NotImplementedError:", exc)

## Round-trip through the writer

A YOLO IC writer is registered, so conversion is real. Because the YOLO
format family serves two tasks (classification and detection), `convert`
needs the task pinned with `task="ic"`.

In [ ]:
from datamaite import convert

ic_rt = ic_root.parent / (ic_root.name + "_rt")
atexit.register(shutil.rmtree, ic_rt, ignore_errors=True)
convert(ic_root, ic_rt, input_format="yolo", output_format="yolo", task="ic")

reloaded = load_ic(ic_rt, dataset_format="yolo")
print(
    "round-trip preserved:",
    reloaded.sample_count == ds.sample_count and reloaded.index2label() == ds.index2label(),
)

## Classification crops from a VisDrone directory

VisDrone static images (`images/` + `annotations/*.txt`) are a detection
format, but `load_ic` can serve classification from the same directory: each
annotated box becomes its own crop sample. The format is writable too:
`write(..., output_format="visdrone")` dispatches on the dataset's task, so a
classification dataset comes back out as a VisDrone-DET split root in which
each crop sample is one annotation row on its source image.

In [ ]:
vd_root = Path(tempfile.mkdtemp(prefix="dm_visdrone_"))
atexit.register(shutil.rmtree, vd_root, ignore_errors=True)
(vd_root / "images").mkdir(parents=True)
(vd_root / "annotations").mkdir()
cv2.imwrite(str(vd_root / "images" / "0000001.jpg"), np.full((60, 80, 3), 120, np.uint8))
# rows: left,top,width,height,score,category,truncation,occlusion (1=pedestrian, 4=car)
(vd_root / "annotations" / "0000001.txt").write_text("10,10,20,15,1,1,0,0\n40,20,25,30,1,4,0,0\n")

ds_vd = load_ic(vd_root, dataset_format="visdrone")
crop, tgt, _ = ds_vd[0]
label = ds_vd.index2label()[int(np.asarray(tgt).argmax())]
print("crop samples:", ds_vd.sample_count, "| first crop", crop.shape, "-> class", label)

In [ ]:
from datamaite import write

vd_ic_out = vd_root.parent / (vd_root.name + "_ic_out")
atexit.register(shutil.rmtree, vd_ic_out, ignore_errors=True)
write(ds_vd, vd_ic_out, output_format="visdrone")  # IC writer: dispatches on ds_vd.task

split_root = vd_ic_out / "VisDrone2019-DET-train"  # the official DET split-root layout
print("\n".join(sorted(str(f.relative_to(vd_ic_out)) for f in vd_ic_out.rglob("*") if f.is_file())))
reloaded_vd = load_ic(split_root, dataset_format="visdrone")
print("round-trip preserved:", reloaded_vd.sample_count == ds_vd.sample_count)

## Where to next

The same VisDrone directory serves detection --
[Working with OD Datasets](Working_with_OD_datasets.ipynb) loads it with
`load_od`. For video datasets, see
[Working with FMV Datasets](Working_with_FMV_datasets.ipynb); for the
loader/writer registry, the
[architecture reference](../reference/architecture.md).

The temp directories are removed when the kernel exits; the cell below
cleans up eagerly.

In [ ]:
for p in (ic_root, ic_rt, vd_root):
    shutil.rmtree(p, ignore_errors=True)
print("removed temp datasets")

<hr>